# Dysarthria Classifier — Training on TORGO (wav2vec2-base)
Run all cells top to bottom. At the end, `dysarthria_head.npz` will be downloaded — place it in `backend/ml_models/`.

In [ ]:
!pip install transformers torch librosa scikit-learn kagglehub -q

In [ ]:
import kagglehub

# Downloads TORGO dataset (~1.75 GB) directly into Colab.
# Your KAGGLE_KEY secret is already set — kagglehub will pick it up automatically.

path = kagglehub.dataset_download("pranaykoppula/torgo-audio")
print("Dataset downloaded to:", path)

ARCHIVE_PATH = path


In [ ]:
import re, os, warnings
import numpy as np
import librosa
from pathlib import Path
from tqdm.notebook import tqdm

os.environ["USE_TF"]   = "0"
os.environ["USE_FLAX"] = "0"

import torch
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import classification_report

warnings.filterwarnings("ignore")

ARCHIVE   = Path(ARCHIVE_PATH)
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
TARGET_SR = 16000
MAX_SAMP  = TARGET_SR * 5   # truncate to 5 s — prevents OOM
HF_MODEL  = "facebook/wav2vec2-base"

# Controls have MC/FC prefix; dysarthric speakers have M/F prefix only
SPEAKER_SEVERITY = {
    # Dysarthric
    "F01": "Severe",   "F03": "Severe",   "F04": "Severe",
    "M01": "Severe",   "M03": "Severe",
    "M02": "Moderate", "M05": "Moderate",
    "M04": "Mild",
    # Controls
    "FC01": "Healthy", "FC02": "Healthy", "FC03": "Healthy",
    "MC01": "Healthy", "MC02": "Healthy", "MC03": "Healthy", "MC04": "Healthy",
}

def get_speaker(folder_name):
    # Matches FC01, MC02, F01, M03, etc. — must be at end of folder name
    m = re.search(r'_([A-Z]{1,2}\d+)(?:S\d+)?$', folder_name)
    return m.group(1) if m else None

files, labels, speakers = [], [], []
for p in sorted(ARCHIVE.rglob("*.wav")):
    sid = get_speaker(p.parent.name)
    sev = SPEAKER_SEVERITY.get(sid) if sid else None
    if sev:
        files.append(p)
        labels.append(sev)
        speakers.append(sid)

import pandas as pd
print(f"Found {len(files)} files across {len(set(speakers))} speakers")
print(pd.Series(labels).value_counts().to_string())

In [ ]:
print(f"Loading {HF_MODEL} ...")
processor = Wav2Vec2Processor.from_pretrained(HF_MODEL)
encoder   = Wav2Vec2Model.from_pretrained(HF_MODEL).eval().to(DEVICE)
print(f"Using device: {DEVICE}")

embeddings = []
for path in tqdm(files, desc="Extracting embeddings"):
    try:
        y, _ = librosa.load(str(path), sr=TARGET_SR, mono=True)
        if len(y) < TARGET_SR * 0.3:
            y = np.zeros(TARGET_SR, np.float32)
    except Exception:
        y = np.zeros(TARGET_SR, np.float32)

    y = y[:MAX_SAMP]  # truncate — prevents OOM
    inp = processor(y, sampling_rate=TARGET_SR, return_tensors="pt")
    with torch.no_grad():
        out = encoder(**{k: v.to(DEVICE) for k, v in inp.items()})
    embeddings.append(out.last_hidden_state.mean(1).squeeze(0).cpu().numpy())

X = np.array(embeddings, dtype=np.float32)
del encoder
print(f"Embeddings shape: {X.shape}")

In [ ]:

# Merge Mild → Moderate for a balanced 3-class problem
labels_3 = ["Moderate" if l == "Mild" else l for l in labels]

le      = LabelEncoder()
y_enc   = le.fit_transform(labels_3)
scaler  = StandardScaler()
X_s     = scaler.fit_transform(X)
groups  = np.array(speakers)

print(f"Classes: {list(le.classes_)}")
print(pd.Series(labels_3).value_counts().to_string())

cv     = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
clf_cv = LogisticRegression(max_iter=1000, C=1.0, class_weight="balanced", random_state=42)
y_pred = cross_val_predict(clf_cv, X_s, y_enc, groups=groups, cv=cv)
print("\n5-fold speaker-independent CV:")
print(classification_report(y_enc, y_pred, target_names=le.classes_))

print("Training final model on full dataset...")
clf = LogisticRegression(max_iter=1000, C=1.0, class_weight="balanced", random_state=42)
clf.fit(X_s, y_enc)

np.savez(
    "dysarthria_head.npz",
    coef         = clf.coef_,
    intercept    = clf.intercept_,
    classes      = le.classes_,
    scaler_mean  = scaler.mean_,
    scaler_scale = scaler.scale_,
)
print("Saved dysarthria_head.npz")


In [ ]:
from google.colab import files
files.download("dysarthria_head.npz")
print("Download started — place the file in backend/ml_models/dysarthria_head.npz")